# 03 · The najiama recipe: digits as categories, dual/triple target encoding, depth-5 LightGBM and CatBoost

**Role in the stack:** the strongest family we trained ourselves. `naji3cb_bag8` (CatBoost, 8 seeds) CV **0.94611**, our best honest
single-model score; `naji3_bag3` (LightGBM V3, 3 seeds) 0.94607; `naji_bag5` (V1, 5 seeds) 0.94590; `naji3xgbn_s42` 0.94597.
Combined weight in the final stack ~0.17 spread over the variants.

## The recipe (najiama, "Pure LGBM", public LB 0.94638)
1. Drop `Number_of_Cars_Owned`.
2. Digits `(x // 10^p) % 10` for p = -4..3 of every numeric column.
3. `*_org_mean`: the label mean of each value in the 10k **original** dataset (97.9% of competition incomes occur there).
4. Every numeric column *also* as a string category (`_cat`), so exact values can be target-encoded.
5. Frequency encoding of every category over train+test.
6. Four spike flags: income == 30,000 (the floor, 9% of rows), income >= 170,537 ("millionaire cliff", 393/393 buyers),
   income in [38k, 42k] ("dead zone"), concern == 1.
7. **In-fold `TargetEncoder`** on all categories, twice with different smoothing (`'auto'` and 10); V3 adds a third (100) and
   four "smooth keys": floor(income), floor(income/100), floor(income/1000), floor(commute).
8. LightGBM depth 5, lr 0.005, colsample 0.3, max_bin 1024, up to 40k rounds.

## The mistake that cost us a week
Our first "replication" changed four things (lr .02, fixed 8000 rounds, bagging on, no `_org_mean`), scored 0.94500, and the
recipe was filed as dead. A red-team audit a week later ran the notebook as published on OUR folds: +0.00046 over the additive
tree, 5/5. The honest port below (only change: early stop on an inner 10% split instead of the scored fold) gives 0.94572 per
seed and lifted the stack from 0.94579 to 0.94607 in one move. **A replication that changes four hyper-parameters is not a
replication.**

In [ ]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, '../src')
from cv import load, run_cv, paired, SEED, NFOLDS, OOF, TEST
X, y, Xte, te_id = load()          # data/train.csv, data/test.csv  (kaggle competitions download -c playground-series-s6e9)
print(X.shape, Xte.shape, 'positives %.4f' % y.mean())

In [ ]:
# Feature build (V3). Verbatim from src/naji3_stream.py minus the experimental VAR probes.
from pathlib import Path
T = 'Will_Buy_EV'
train = pd.read_csv('../data/train.csv'); test = pd.read_csv('../data/test.csv')
orig = pd.read_csv('../data/EV_Adoption_and_Range_Anxiety_Dataset.csv')      # kaggle datasets download itzzomkar/ev-adoption-behavior-and-range-anxiety
train[T] = train[T].map({'Yes': 1, 'No': 0}); orig[T] = orig[T].map({'Yes': 1, 'No': 0})
train['is_train'] = 1; test['is_train'] = 0; test[T] = np.nan
c = pd.concat([train, test], ignore_index=True).drop(columns=['Number_of_Cars_Owned'])
cat_cols = c.select_dtypes(include=['object', 'string']).columns.tolist()
num_cols = [k for k in c.columns if k not in cat_cols + ['id', 'is_train', T]]
for k in list(num_cols):
    for p in range(-4, 4):
        c[f'{k}_digit{p}'] = (c[k].fillna(0) // (10 ** p) % 10).astype('int8'); num_cols.append(f'{k}_digit{p}')
og = orig[T].mean()
for k in cat_cols + num_cols:
    if k in orig.columns:
        c[f'{k}_org_mean'] = c[k].map(orig.groupby(k, observed=False)[T].mean()).fillna(og).astype(float)
n2c = []
for k in num_cols:
    c[f'{k}_cat'] = c[k].fillna('NaN').astype(str); n2c.append(f'{k}_cat')
all_cats = cat_cols + n2c
c['is_30k_spike'] = (c['Annual_Income_USD'] == 30000.0).astype('int8')
c['is_millionaire_cliff'] = (c['Annual_Income_USD'] >= 170537.0).astype('int8')
c['is_dead_zone'] = ((c['Annual_Income_USD'] >= 38000.0) & (c['Annual_Income_USD'] <= 42000.0)).astype('int8')
c['is_env_hater'] = (c['Environmental_Concern_Level'] == 1).astype('int8')
# V3 addition 1: smooth keys
c['income_exact_int'] = np.floor(c['Annual_Income_USD']).astype(str)
c['income100_floor'] = np.floor(c['Annual_Income_USD'] / 100.0).astype(str)
c['income1000_floor'] = np.floor(c['Annual_Income_USD'] / 1000.0).astype(str)
c['commute_integer'] = np.floor(c['Daily_Commute_km']).astype(str)
SMOOTH = ['income_exact_int', 'income100_floor', 'income1000_floor', 'commute_integer']
all_cats += SMOOTH
for k in all_cats:
    c[f'{k}_fe'] = c[k].map(c[k].value_counts(normalize=True)).astype(float).fillna(0.0)
train = c[c.is_train == 1].drop(columns=['is_train']); test = c[c.is_train == 0].drop(columns=['is_train', T])
# drop perfectly-correlated and constant columns (same filter as the notebook)
ev = [k for k in train.columns if k not in ['id', T] and pd.api.types.is_numeric_dtype(train[k])]
cm = train[ev].corr().abs(); up = cm.where(np.triu(np.ones(cm.shape), k=1).astype(bool))
drop = set(k for k in up.columns if any(up[k] == 1.0)) | set(k for k in train.columns if train[k].nunique() == 1) \
       | set(k for k in test.columns if test[k].nunique() == 1)
drop = [k for k in drop if k not in ['id', T]]
train = train.drop(columns=drop); test = test.drop(columns=drop)
FEAT = [k for k in test.columns if k != 'id']; TEC = [k for k in all_cats if k not in drop]
print('features %d, target-encoded cats %d, dropped %d' % (len(FEAT), len(TEC), len(drop)))
Xn, yn, Xtn = train[FEAT], train[T].astype(int), test[FEAT]

In [ ]:
from sklearn.preprocessing import TargetEncoder

def encode(Xa, ya, others):
    """Triple in-fold TE (smooth auto / 10 / 100). Train rows get cross-fitted values (cv=5); other frames get full-fit values."""
    encs = [('auto', TargetEncoder(shuffle=True, cv=5, smooth='auto', random_state=SEED)),
            ('10', TargetEncoder(shuffle=True, cv=5, smooth=10.0, random_state=SEED)),
            ('100', TargetEncoder(shuffle=True, cv=5, smooth=100.0, random_state=SEED))]
    A = [e.fit_transform(Xa[TEC], ya) for _, e in encs]
    def build(F, mats):
        F = F.drop(columns=TEC).copy()
        for (lab, _), M in zip(encs, mats):
            for i, k in enumerate(TEC): F[f'{k}_TE_{lab}'] = M[:, i].astype('float32')
        return F
    return build(Xa, A), [build(F, [e.transform(F[TEC]) for _, e in encs]) for F in others]

In [ ]:
import lightgbm as lgb, catboost as cb
# LightGBM = the published V3 parameters. CatBoost = kospintr's public single-CatBoost parameters (LB 0.94603) on the same features.
P_LGB = dict(n_estimators=40000, learning_rate=0.005, max_depth=5, num_leaves=32, min_child_samples=10, subsample=0.8,
             colsample_bytree=0.3, reg_alpha=0.071, reg_lambda=2.0, max_bin=1024, feature_pre_filter=False,
             metric='auc', n_jobs=-1, verbose=-1)
P_CB = dict(loss_function='Logloss', eval_metric='AUC', learning_rate=0.025, depth=5, max_bin=1152, l2_leaf_reg=3.0,
            auto_class_weights='Balanced', iterations=40000, task_type='GPU', verbose=0)
MODEL, seed = 'cb', 42      # 'lgb' -> naji3_s<seed>, 'cb' -> naji3cb_s<seed>

def fit(Xa, ya, Xb=None, yb=None, rounds=None):
    if MODEL == 'lgb':
        m = lgb.LGBMClassifier(**dict(P_LGB, random_state=seed, **({'n_estimators': rounds} if rounds else {}))).fit(
            Xa, ya, eval_set=None if rounds else [(Xb, yb)], callbacks=None if rounds else [lgb.early_stopping(700, verbose=False)])
        return m, rounds or m.best_iteration_
    m = cb.CatBoostClassifier(**dict(P_CB, random_seed=seed, **({'iterations': rounds} if rounds else {}))).fit(
        Xa, ya, eval_set=None if rounds else (Xb, yb), early_stopping_rounds=None if rounds else 150)
    return m, rounds or m.get_best_iteration()

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
TAG = 'naji3%s_s%d' % ('' if MODEL == 'lgb' else MODEL, seed)
oof, tp, iters = np.zeros(len(Xn)), np.zeros(len(Xtn)), []
for f, (itr, iva) in enumerate(StratifiedKFold(NFOLDS, shuffle=True, random_state=SEED).split(Xn, yn)):
    Xtr, ytr = Xn.iloc[itr], yn.iloc[itr]
    ia, ib = train_test_split(np.arange(len(itr)), test_size=0.1, stratify=ytr, random_state=seed)   # inner ES split
    Xa, (Xb, Xva, Xte_f) = encode(Xtr.iloc[ia], ytr.iloc[ia], [Xtr.iloc[ib], Xn.iloc[iva], Xtn])
    m, it = fit(Xa, ytr.iloc[ia], Xb, ytr.iloc[ib])
    oof[iva] = m.predict_proba(Xva)[:, 1]; tp += m.predict_proba(Xte_f)[:, 1] / NFOLDS; iters.append(it)
    print('%s fold%d=%.5f iters=%d' % (TAG, f, roc_auc_score(yn.iloc[iva], oof[iva]), it))
print('%s oof=%.5f' % (TAG, roc_auc_score(yn, oof)))
np.save(OOF / f'oof_{TAG}.npy', oof); np.save(TEST / f'test_{TAG}.npy', tp)
# full-data refit: TE on all train rows, fixed rounds = 1.1 x mean early-stopped iterations
Xa, (Xte_f,) = encode(Xn, yn, [Xtn]); m, _ = fit(Xa, yn, rounds=int(np.mean(iters) * 1.1))
np.save(TEST / f'test_{TAG}_full.npy', m.predict_proba(Xte_f)[:, 1])

## Results on our folds (seed 42 unless noted)
| variant | CV | note |
|---|---|---|
| V1 LightGBM (`naji_stream.py`), 5-seed bag | 0.94590 | 0.94572 per seed; the first +0.0003 stack lift |
| V3 LightGBM, 3-seed bag | 0.94607 | smooth keys + third TE |
| **V3 CatBoost, 8-seed bag** | **0.94611** | best honest single we trained |
| V3 XGBoost, najiama/Tilii params (depth 4 lossguide, gamma 3.67) | 0.94597 | +0.00016 over our own XGB params, 5/5 |
| Optuna on the V1 LightGBM, 99 trials | +0.00001 | the author's settings were already optimal |
| Optuna on CatBoost, 151 trials | +0.00001 | winner's curse exactly as predicted |

## What we learned
- The author's reported 0.94606 with early stopping on the scored fold = 0.94592 honest. Always re-measure a public CV under your
  own protocol before you compare it to yours.
- Parameters copied from one library's winner do not transfer: najiama's XGB parameters beat ours by +0.00016 on identical features.
- Everything else on this family measured zero: key ladders, pair TEs, TE seed averaging, local-rate kernels, 16k CatBoost borders.